# Meta-heurísticas - 2026/2
## Atividade T1

### Gerador de Instâncias



In [1]:
import random
import os

def gerar_instancia_com_intersecao(n_pessoas, nome_arquivo):
    # 1. Definir o tamanho do núcleo (ex: 20% das pessoas, mínimo de 4)
    tamanho_nucleo = max(4, int(n_pessoas * 0.2))
    if tamanho_nucleo % 2 != 0:
        tamanho_nucleo += 1 # Garante que seja par para facilitar a divisão

    pessoas = list(range(1, n_pessoas + 1))
    random.shuffle(pessoas)

    nucleo = pessoas[:tamanho_nucleo]
    restantes = pessoas[tamanho_nucleo:]

    saldos = {p: 0 for p in pessoas}

    # 2. Atribuir saldos ao núcleo para que a soma de todos eles seja ZERO
    soma_nucleo = 0
    for p in nucleo[:-1]:
        valor = random.randint(-500, 500)
        while valor == 0:
            valor = random.randint(-500, 500)
        saldos[p] = valor
        soma_nucleo += valor
    # O último do núcleo absorve a diferença para zerar o grupo
    saldos[nucleo[-1]] = -soma_nucleo 

    # 3. Dividir o núcleo em pares (subgrupos)
    k_grupos = tamanho_nucleo // 2
    subgrupos_nucleo = [nucleo[i:i + 2] for i in range(0, tamanho_nucleo, 2)]

    # 4. Dividir as pessoas "restantes" em 'k_grupos' de forma equilibrada
    subgrupos_restantes = [[] for _ in range(k_grupos)]
    for i, p in enumerate(restantes):
        subgrupos_restantes[i % k_grupos].append(p)

    # 5. Fazer a mágica da interseção: o (subgrupo do nucleo) + (subgrupo restante) deve somar ZERO
    for i in range(k_grupos):
        soma_sub_nucleo = sum(saldos[p] for p in subgrupos_nucleo[i])
        grupo_restante = subgrupos_restantes[i]

        if not grupo_restante:
            continue

        soma_temp_restante = 0
        # Gera saldos aleatórios para quase todos do grupo restante
        for p in grupo_restante[:-1]:
            valor = random.randint(-500, 500)
            while valor == 0:
                valor = random.randint(-500, 500)
            saldos[p] = valor
            soma_temp_restante += valor

        # A peça chave: O último integrante do grupo restante assume a dívida exata 
        # para que a soma do (sub_nucleo + restante) zere.
        saldos[grupo_restante[-1]] = -(soma_sub_nucleo + soma_temp_restante)

    # 6. Remover quem ficou com zero e embaralhar a lista final
    pessoas_info = [{"id": p, "saldo": saldos[p]} for p in pessoas if saldos[p] != 0]
    pessoas_info.sort(key=lambda x: x["id"])

    # 7. Salvar no arquivo
    with open(nome_arquivo, 'w') as f:
        for pessoa in pessoas_info:
            f.write(f"{pessoa['id']} {pessoa['saldo']}\n")
            
    # Salvar metadados para você conseguir analisar a qualidade da sua busca local depois
    os.makedirs("data/metadados", exist_ok=True)
    nome_base = os.path.basename(nome_arquivo)
    nome_arquivo_meta = f"data/metadados/{nome_base.replace('.txt', '_metadados.txt')}"
    
    with open(nome_arquivo_meta, 'w') as f:
        f.write("=== METADADOS (GABARITO DA INSTÂNCIA) ===\n")
        f.write(f"Tamanho Núcleo: {tamanho_nucleo} | Quantidade de Grupos Ótimos: {k_grupos}\n")
        f.write(f"-> O algoritmo Guloso tende a errar gerando {n_pessoas - 2} transações.\n")
        f.write(f"-> O Ótimo Global plantado é de {n_pessoas - k_grupos} transações.\n")
        f.write("=========================================\n")

In [2]:
valores = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400, 420, 440, 460, 480, 500]
random.seed(42)

os.makedirs("data/instancias", exist_ok=True)

for numeros in valores:
    caminho_arquivo = f"data/instancias/instancia_splitwise_{numeros}.txt"
    gerar_instancia_com_intersecao(numeros, caminho_arquivo)

### Leitura dos arquivos

In [123]:
import pandas as pd
import time

def ler_instancia_para_df(nome_arquivo, n_pessoas):
    df = pd.read_csv(nome_arquivo, sep=" ", names=["id", "saldo"])
    return df

In [130]:
instancias = {}
tamanhos = [
    "9", "20", "30", "50", "50_8", "100", "100_20", "150", "150_15",
    "200", "200_8", "200_80", "300", "300_60", "400", "400_35", "500",
    "500_25", "500_188", "600", "600_40", "700", "700_90", "800", "800_80",
    "900", "900_100", "1000", "1000_120", "1100", "1300", "1500"
]

for tamanho in tamanhos:
    caminho_arquivo = f"data/instancias-turma/instancia_splitwise_{tamanho}.txt"
    df_carregado = ler_instancia_para_df(caminho_arquivo, tamanho)

    instancias[tamanho] = df_carregado

### Iniciando busca de solução

#### Algoritmo Construtivo
*    Guloso



In [125]:
def construtivo_guloso(df_instancia):
    df_temp = df_instancia.copy()
    inicio_tempo = time.perf_counter()

    credores = df_temp[df_temp['saldo'] > 0].values.tolist()
    devedores = df_temp[df_temp['saldo'] < 0].values.tolist()
    transacoes = []

    while credores and devedores:
        devedores.sort(key=lambda x: abs(x[1]), reverse=True)
        credores.sort(key=lambda x: x[1], reverse=True)

        match_encontrado = False

        mapa_credores = {cred[1]: j for j, cred in enumerate(credores)}

        for i, dev in enumerate(devedores):
            saldo_pendente = abs(dev[1])

            if saldo_pendente in mapa_credores:
                j = mapa_credores[saldo_pendente]
                cred = credores[j]

                transacoes.append({
                    'devedor': int(dev[0]),
                    'credor': int(cred[0]),
                    'valor': cred[1]
                })

                devedores.pop(i)
                credores.pop(j)
                match_encontrado = True
                break

        if not match_encontrado:
            dev = devedores[0]
            cred = credores[0]

            valor_transacao = min(abs(dev[1]), cred[1])

            transacoes.append({
                'devedor': int(dev[0]),
                'credor': int(cred[0]),
                'valor': valor_transacao
            })

            devedores[0][1] += valor_transacao
            credores[0][1] -= valor_transacao

            if devedores[0][1] == 0:
                devedores.pop(0)
            if credores[0][1] == 0:
                credores.pop(0)

    fim_tempo = time.perf_counter()
    tempo_execucao = fim_tempo - inicio_tempo

    return transacoes, tempo_execucao

*    Aleatório


In [126]:
def construtivo_randomizado(df_instancia, alpha=0.2):
    df_temp = df_instancia.copy()
    inicio_tempo = time.perf_counter()

    credores = df_temp[df_temp['saldo'] > 0].values.tolist()
    devedores = df_temp[df_temp['saldo'] < 0].values.tolist()

    transacoes = []

    while credores and devedores:
        devedores.sort(key=lambda x: x[1])
        credores.sort(key=lambda x: x[1], reverse=True)

        limite_dev = max(1, int(alpha * len(devedores)))
        idx_dev = random.randrange(limite_dev)
        dev = devedores[idx_dev]
        magnitude_dev = -dev[1]

        idx_cred = -1
        for j, cred in enumerate(credores):
            if cred[1] == magnitude_dev:
                idx_cred = j
                break

        if idx_cred == -1:
            limite_cred = max(1, int(alpha * len(credores)))
            idx_cred = random.randrange(limite_cred)

        cred = credores[idx_cred]

        valor_transacao = min(magnitude_dev, cred[1])
        transacoes.append({
            'devedor': int(dev[0]),
            'credor': int(cred[0]),
            'valor': valor_transacao
        })

        devedores[idx_dev][1] += valor_transacao
        credores[idx_cred][1] -= valor_transacao

        if devedores[idx_dev][1] == 0:
            devedores.pop(idx_dev)
        if credores[idx_cred][1] == 0:
            credores.pop(idx_cred)


    fim_tempo = time.perf_counter()
    tempo_execucao = fim_tempo - inicio_tempo

    return transacoes, tempo_execucao

#### Algoritmo Busca Local

##### Funções Auxiliares

In [131]:
def vizinhanca_destruicao_reconstrucao(solucao_atual, taxa_destruicao=0.2):
    # 1. Contar a frequência de cada pessoa nas transações
    frequencia = {}
    for t in solucao_atual:
        frequencia[t['devedor']] = frequencia.get(t['devedor'], 0) + 1
        frequencia[t['credor']] = frequencia.get(t['credor'], 0) + 1

    # 2. Ordenar pessoas pelo número de transações (do maior para o menor)
    pessoas_ordenadas = sorted(frequencia.keys(), key=lambda x: frequencia[x], reverse=True)

    # 3. Selecionar o alvo com um leve grau de aleatoriedade para não estagnar
    qtd_selecionar = max(3, int(len(pessoas_ordenadas) * taxa_destruicao))
    
    # Pegamos os 40% piores e sorteamos 'qtd_selecionar' pessoas de dentro desse grupo
    pool_candidatos = pessoas_ordenadas[:max(qtd_selecionar * 2, 4)]
    pessoas_alvo = set(random.sample(pool_candidatos, min(qtd_selecionar, len(pool_candidatos))))

    # 4. Separar as transações que envolvem essas pessoas
    transacoes_restantes = []
    transacoes_destruidas = []
    
    for t in solucao_atual:
        # Se qualquer uma das pontas da transação for um alvo, ela é destruída
        if t['devedor'] in pessoas_alvo or t['credor'] in pessoas_alvo:
            transacoes_destruidas.append(t)
        else:
            transacoes_restantes.append(t)

    # 5. Calcular o saldo LOCAL exato a partir das transações rompidas
    saldos_locais = {}
    for t in transacoes_destruidas:
        saldos_locais[t['devedor']] = saldos_locais.get(t['devedor'], 0) - t['valor']
        saldos_locais[t['credor']] = saldos_locais.get(t['credor'], 0) + t['valor']

    # Criar um DataFrame no mesmo formato que sua função gulosa espera
    df_local = pd.DataFrame(
        [{'id': p, 'saldo': s} for p, s in saldos_locais.items() if s != 0]
    )

    if df_local.empty:
        return solucao_atual
        
    # 6. Reconstrução usando sua função já existente
    # Precisamos renomear a coluna de id/saldo conforme seu construtor ou ajustar lá
    # O df deve ter colunas [0, 1] ou ['id', 'saldo'] conforme implementado no ler_instancia_para_df
    df_local.columns = ['id', 'saldo'] 
    
    novas_transacoes, _ = construtivo_guloso(df_local)

    # 7. Avaliação
    if len(novas_transacoes) < len(transacoes_destruidas):
        # Houve melhoria
        return transacoes_restantes + novas_transacoes
    else:
        # Não houve melhoria, devolvemos a solução intacta
        return solucao_atual

*    First-improvement

In [132]:
import time

def bl_first_improvement(solucao_inicial, max_tentativas=500):
    solucao_atual = solucao_inicial.copy()
    inicio_tempo = time.perf_counter()
    
    n_it = 0 # Contador de iterações exigido pela Tabela 1
    tentativas_sem_melhora = 0
    
    while tentativas_sem_melhora < max_tentativas:
        n_it += 1
        
        # Gera UM vizinho aplicando o movimento de destruição e reconstrução
        vizinho = vizinhanca_destruicao_reconstrucao(solucao_atual)
        
        # Critério First-Improvement: Aceita a primeira melhora imediata
        if len(vizinho) < len(solucao_atual):
            solucao_atual = vizinho
            tentativas_sem_melhora = 0 # Zera o contador, pois achou uma melhora
        else:
            tentativas_sem_melhora += 1 # Não melhorou, incrementa o contador de falhas
            
    tempo_total = time.perf_counter() - inicio_tempo
    
    return solucao_atual, tempo_total, n_it

*    Best-improvement

### RESULTADOS FINAIS

In [ ]:
resultados_tabela = []

for tamanho, df_inst in instancias.items():
    # 1. Executa Algoritmos Construtivos
    transacoes_g, tempo_g = construtivo_guloso(df_inst)
    transacoes_r, tempo_r = construtivo_randomizado(df_inst)

    # 2. Executa First-Improvement sobre as duas soluções iniciais
    sol_fi_g, t_fi_g, nit_fi_g = bl_first_improvement(transacoes_g)
    sol_fi_r, t_fi_r, nit_fi_r = bl_first_improvement(transacoes_r)

    # 3. Executa Best-Improvement sobre as duas soluções iniciais
    # sol_bi_g, t_bi_g, nit_bi_g = bl_best_improvement(transacoes_g)
    # sol_bi_r, t_bi_r, nit_bi_r = bl_best_improvement(transacoes_r)

    # 4. Grava os resultados consolidados
    resultados_tabela.append({
        'I': f"Inst_{tamanho}",

        'AC_G (sol)': len(transacoes_g),
        'AC_G (t(s))': round(tempo_g, 6),
        'AC_R (sol)': len(transacoes_r),
        'AC_R (t(s))': round(tempo_r, 6),

        'BL_FI(AC_G) (sol)': len(sol_fi_g),
        'BL_FI(AC_G) (t(s))': round(t_fi_g, 6),
        'BL_FI(AC_G) (N_It)': nit_fi_g,

        'BL_FI(AC_R) (sol)': len(sol_fi_r),
        'BL_FI(AC_R) (t(s))': round(t_fi_r, 6),
        'BL_FI(AC_R) (N_It)': nit_fi_r,

        # 'BL_BI(AC_G) (sol)': len(sol_bi_g),
        # 'BL_BI(AC_G) (t(s))': round(t_bi_g, 6),
        # 'BL_BI(AC_G) (N_It)': nit_bi_g,

        # 'BL_BI(AC_R) (sol)': len(sol_bi_r),
        # 'BL_BI(AC_R) (t(s))': round(t_bi_r, 6),
        # 'BL_BI(AC_R) (N_It)': nit_bi_r
    })

df_resultados_parciais = pd.DataFrame(resultados_tabela)
print(df_resultados_parciais.to_string())